[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agustrionopj/python-mini-workshop/blob/main/ccd_preprocessing_astropy.ipynb?forcePlay=true)

# Pra-proses data CCD dengan `Astropy`

**Mini-workshop:** _Pelatihan Python — Imahnoong_  
**Pemateri:** Agus T. P. Jatmiko (Observatorium Bosscha)

**Tanggal:** 2026-10-31

---

<!-- > **TODO (before publishing):** replace `YOURUSER/YOURREPO` in the badge URL
> above and in the data-download cell (Section 1). There are exactly **2 spots**. -->

## ⚠️ Jangan edit file ini secara langsung!

Anda membuka file ini dari  **file asli** pada GitHub.
Gunakan dalam mode _read-only_:

- Anda bisa menjalankan seluruh _cell_ secara bebas dan mandiri.
- Jika ingin melakukan perubahan, lakukan hal berikut: `File → Save a copy in Drive` (simpan ke Drive anda sendiri).
- File asli pada repository tidak akan melihat perubahan yang anda lakukan.

Semua proses akan berlangsung di dalam Colab.

## 0. Tujuan Workshop

Setelah workshop ini anda akan dapat:

1. Membaca file FITS dan inspeksi header-nya dengan `astropy.io.fits`.
2. Membuat **master bias**, **master dark**, dan **masterflat**.
3. Memahami kenapa sebuah pilihan parameter dibuat/dilakukan.
4. Kalibrasi citra SCIENCE.
5. Lakukan hal yang sama dengan `ccdproc`.


In [2]:
# --- Setup: install paket yang diperlukan (Colab bersifat sementara, jadi jalankan ini setiap sesi) ---
# !pip install -q astropy ccdproc pandas numpy matplotlib

In [5]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.stats import mad_std
import ccdproc
from astroscrappy import detect_cosmics

plt.rcParams['figure.dpi'] = 100

# Central place for paths used throughout the notebook
DATA_DIR = 'data'

os.makedirs(DATA_DIR, exist_ok=True)
print('Ready. Astropy version:', __import__('astropy').__version__)

Ready. Astropy version: 7.0.0


## 1. Get the data

The calibration frames are stored in the workshop GitHub repository, grouped
by type. We fetch them straight into this Colab runtime.

> **TODO (before publishing):** set `DATA_BASE_URL` to your real repo,
> and adjust the file lists to the actual filenames you committed.
# Dataset utama (TANPA overscan) — dipakai oleh Bagian 2–9
# pola nama file: <jenis>-mock-<eksposur>-<nomor>.fit
BIAS_FILES = [f"mock_nooverscan/bias/bias-mock-{i:03d}.fit" for i in range(1, 11)]

DARK_FILES = ([f"mock_nooverscan/dark/dark-mock-0002s-{i:03d}.fit" for i in range(1, 6)]
            + [f"mock_nooverscan/dark/dark-mock-0120s-{i:03d}.fit" for i in range(1, 6)])

FLAT_FILES = [f"mock_nooverscan/flat/flat-mock-v-2s-{i:03d}.fit" for i in range(1, 11)]

SCIENCE_FILES = [f"mock_nooverscan/science/field-mock-v-120s-{i:03d}.fit" for i in range(1, 6)]

# --- Dataset kedua: kamera DENGAN strip overscan (dipakai di Bagian 3B) ---
BIAS_OVERSCAN_FILES    = [f"mock_overscan/bias/bias-mock-{i:03d}.fit" for i in range(1, 11)]
DARK_OVERSCAN_FILES    = ([f"mock_overscan/dark/dark-mock-0002s-{i:03d}.fit" for i in range(1, 6)]
                        + [f"mock_overscan/dark/dark-mock-0120s-{i:03d}.fit" for i in range(1, 6)])
FLAT_OVERSCAN_FILES    = [f"mock_overscan/flat/flat-mock-v-2s-{i:03d}.fit" for i in range(1, 11)]
SCIENCE_OVERSCAN_FILES = [f"mock_overscan/science/field-mock-v-120s-{i:03d}.fit" for i in range(1, 6)]

ALL_FILES = (BIAS_FILES + DARK_FILES + FLAT_FILES + SCIENCE_FILES
           + BIAS_OVERSCAN_FILES + DARK_OVERSCAN_FILES
           + FLAT_OVERSCAN_FILES + SCIENCE_OVERSCAN_FILES)
print(f"Total files to fetch: {len(ALL_FILES)}")


In [ ]:
# ---------------------------------------------------------------------------
# TODO: replace YOURUSER/YOURREPO with the real GitHub account/repo name
# ---------------------------------------------------------------------------
DATA_BASE_URL = "https://raw.githubusercontent.com/agustrionopj/python-mini-workshop/main/data"

# TODO: fill with your actual filenames, matching what you committed to the repo
BIAS_FILES = [
    "bias/bias-b2-001.fit",
    "bias/bias-b2-002.fit",
    "bias/bias-b2-003.fit",
    "bias/bias-b2-004.fit",
    "bias/bias-b2-005.fit",
    # ...
]

DARK_FILES = [
    "dark/dark-b2-0002s--001.fit",
    "dark/dark-b2-0002s--002.fit",
    "dark/dark-b2-0002s--003.fit",
    "dark/dark-b2-0002s--004.fit",
    "dark/dark-b2-0002s--005.fit",
    "dark/dark-b2-120s-001.fit",
    "dark/dark-b2-120s-002.fit",
    "dark/dark-b2-120s-003.fit",
    "dark/dark-b2-120s-004.fit",
    "dark/dark-b2-120s-005.fit",
    # ...
]

FLAT_FILES = [
    "flat/flat-b2-v-2s-001.fit",
    "flat/flat-b2-v-2s-002.fit",
    "flat/flat-b2-v-2s-003.fit",
    "flat/flat-b2-v-2s-004.fit",
    "flat/flat-b2-v-2s-005.fit",
    # ...
]

SCIENCE_FILES = [
    "science/field-b2-v-120s-002.fit",
    # ...
]

ALL_FILES = BIAS_FILES + DARK_FILES + FLAT_FILES + SCIENCE_FILES
print(f"Total files to fetch: {len(ALL_FILES)}")

In [ ]:
# Download the files (keeps the repo as the single source of truth)
for relpath in ALL_FILES:
    lastrun = os.path.join(DATA_DIR, relpath)
    os.makedirs(os.path.dirname(lastrun), exist_ok=True)
    if not os.path.exists(lastrun):   # simple cache: skip if already fetched
        !wget -q -O {lastrun} {DATA_BASE_URL}/{relpath.replace(' ', '%20')}
        print("fetched:", relpath)

print("\nFiles on disk:")
!find {DATA_DIR} -name "*.fit*" | sort

## 2. Inspect a raw frame

Before combining anything, always look at one frame and its header.
Key header keywords to check: `IMAGETYP` (frame type), `EXPTIME`,
`DATE-OBS`, `GAIN`, `SET-TEMP`.

**DISCUSS:** What should a pure bias frame look like? (No structure, no stars — just the readout pattern.)

In [ ]:
# Open the first science frame and look at its header
# TODO: pick a real filename from your list
SAMPLE = os.path.join(DATA_DIR, SCIENCE_FILES[0])

with fits.open(SAMPLE) as hdul:
    hdul.info()
    header = hdul[0].header.copy()
    data  = hdul[0].data.astype(float)

header  # last expression = nice HTML table in Colab

In [ ]:
def show_image(img, title="", ax=None):
    """Display with a percentile stretch (cheap equivalent of DS9's zscale)."""
    lo, hi = np.percentile(img, (5, 95))
    show = ax.imshow(img, cmap='gray', origin='lower', vmin=lo, vmax=hi)
    ax.set_title(title)
    return show

fig, ax = plt.subplots(figsize=(8, 8))
show_image(data, "RAW science frame", ax)
plt.show()

print(f"shape: {data.shape}")
print(f"basic stats:  min={data.min():.1f}  max={data.max():.1f}  "
      f"median={np.median(data):.1f}  MAD-std={mad_std(data):.1f}")

# TODO: guide participants — what do the hot columns, the glow in a corner,
# and the speckles (cosmic rays / stars) tell you about this detector?

## 3. Master Bias

Median-combining the bias frames rejects cosmic-ray hits and keeps the
fixed readout pattern (note: bias has no overscan trimming here —
TODO: add an overscan section if your camera has one).

**Why median?** A single cosmic ray in one outlier frame is discarded;
a mean would be dragged by it.

In [ ]:
def load_fits_list(file_list):
    """Load a list of FITS files into a stack of 2D float arrays."""
    frames = []
    for relpath in file_list:
        with fits.open(os.path.join(DATA_DIR, relpath)) as hdul:
            frames.append(hdul[0].data.astype(float))
    return np.array(frames)

# --- Master bias — median combine ----------------------------------------
bias_stack = load_fits_list(BIAS_FILES)              # TODO: confirm shapes match!
print("bias stack:", bias_stack.shape)

master_bias = np.median(bias_stack, axis=0)

fig, ax = plt.subplots(figsize=(6, 6))
show_image(master_bias, "MASTER BIAS", ax)
plt.show()

print(f"master bias: median={np.median(master_bias):.1f}  MAD-std={mad_std(master_bias):.1f}")

# TODO (exercise): compute master_bias with mean instead of median.
# Compare the MAD-std of both — which set of results do you prefer, and why?

## 3B. Overscan — koreksi level bias per frame

Kamera tertentu (ditiru oleh dataset `mock_overscan`) membaca chip
melebihi area citra: **strip overscan** berupa beberapa kolom tambahan di
sisi kanan. Strip ini tidak menerima cahaya maupun arus gelap — isinya
hanyalah **level bias** yang dibaca bersamaan dengan setiap baris citra.

Kata kunci header yang relevan:

- `DATASEC` — area citra sebenarnya (bagian yang "berguna"),
- `BIASSEC` — strip overscan yang dipakai mengukur level bias per baris,
- `TRIMSEC` — area yang dipertahankan setelah trimming.

Kenapa penting? Level bias bisa **bergeser antar frame** (drift suhu /
sumber daya). Dengan overscan kita dapat mengoreksi level per-baris
**untuk setiap frame** — sesuatu yang tidak bisa dilakukan master bias saja.

**Latihan:**
1. Bandingkan level bias (dari `BIASSEC`) frame ke-1 dan ke-10 — apakah
   bergeser?
2. Sebutkan kondisi kapan koreksi overscan lebih baik daripada hanya
   master bias, dan sebaliknya.

In [ ]:
# --- Inspeksi frame dengan overscan --------------------------------------
OS_SAMPLE = os.path.join(DATA_DIR, BIAS_OVERSCAN_FILES[0])

with fits.open(OS_SAMPLE) as hdul:
    hdr_os = hdul[0].header.copy()
    img_os = hdul[0].data.astype(float)

print("shape:", img_os.shape, "  (baris, kolom)")
for k in ('DATASEC', 'BIASSEC', 'TRIMSEC'):
    print(f"  {k:8s} = {hdr_os.get(k)}")

fig, ax = plt.subplots(figsize=(10, 5))
show_image(img_os, "BIAS dengan strip overscan (sisi kanan)", ax)
plt.show()

In [ ]:
# --- Manual (numpy): ukur level bias per baris, koreksi, lalu trim -------
def parse_sec(text):
    """'[513:528,1:512]' -> (slice baris, slice kolom) gaya numpy."""
    kolom, baris = text.strip('[]').split(',')
    x1, x2 = map(int, kolom.split(':'))
    y1, y2 = map(int, baris.split(':'))
    return slice(y1 - 1, y2), slice(x1 - 1, x2)

datasec = parse_sec(hdr_os['DATASEC'])
biassec = parse_sec(hdr_os['BIASSEC'])

bias_per_row = np.median(img_os[biassec], axis=1)        # 1 nilai per baris
print(f"level bias per baris: median={np.median(bias_per_row):.2f}  "
      f"min={bias_per_row.min():.2f}  max={bias_per_row.max():.2f}")

img_os_corr = img_os - bias_per_row[:, np.newaxis]       # koreksi per baris
img_os_trim = img_os_corr[datasec]                       # buang strip overscan

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(bias_per_row)
axes[0].set_xlabel("baris")
axes[0].set_ylabel("ADU")
axes[0].set_title("level bias per baris (dari strip overscan)")

baris_tengah = img_os[img_os.shape[0] // 2, :]
axes[1].plot(baris_tengah)
n_os = img_os[biassec].shape[1]              # lebar strip overscan
axes[1].axvspan(img_os.shape[1] - n_os, img_os.shape[1],
                color='r', alpha=0.2)
axes[1].set_xlabel("kolom")
axes[1].set_title("profil satu baris (merah = strip overscan)")
plt.show()

In [ ]:
# --- Cara "bawaan": ccdproc.subtract_overscan + trim_image ---------------
ccd = ccdproc.CCDData.read(OS_SAMPLE, unit='adu')

ccd_os = ccdproc.subtract_overscan(
    ccd,
    fits_section=ccd.header['BIASSEC'],
    median=True,
    overscan_axis=1,
)
ccd_os = ccdproc.trim_image(ccd_os, fits_section=ccd.header['DATASEC'])

print("shape setelah trim:", ccd_os.data.shape)

diff = img_os_trim - ccd_os.data
print(f"selisih manual vs ccdproc: max|d| = {np.abs(diff).max():.3e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show_image(img_os_trim, "trim (manual numpy)", axes[0])
show_image(ccd_os.data, "trim (ccdproc)", axes[1])
plt.show()

## 4. Master Dark

Current accumulates roughly linearly with exposure time, so we combine a
set of darks of equal exposure and later **scale** by the science exposure.
For workshop simplicity we use the same exposure time as the science frames.

**DISCUSS:** How would you handle science frames of a different exposure?

In [ ]:
# --- Master dark — median combine, bias-subtracted -----------------------
# PENTING: dark dengan EXPTIME berbeda tidak boleh digabung langsung.
# Kelompokkan dulu berdasarkan EXPTIME, lalu pilih yang paling dekat
# dengan EXPTIME frame science (header sudah dibaca di Bagian 2).
def read_exptime(relpath):
    with fits.open(os.path.join(DATA_DIR, relpath)) as hdul:
        return float(hdul[0].header['EXPTIME'])

dark_groups = {}
for relpath in DARK_FILES:
    dark_groups.setdefault(read_exptime(relpath), []).append(relpath)
print("kelompok dark per EXPTIME:", {t: len(v) for t, v in dark_groups.items()})

sci_exptime = float(header['EXPTIME'])
dark_exptime = min(dark_groups, key=lambda t: abs(t - sci_exptime))
print(f"memakai kelompok dark EXPTIME = {dark_exptime} s (science = {sci_exptime} s)")

dark_stack = load_fits_list(dark_groups[dark_exptime])
print("dark stack:", dark_stack.shape)

master_dark_raw = np.median(dark_stack, axis=0)
master_dark     = master_dark_raw - master_bias      # ADU pada eksposur dark_exptime

fig, ax = plt.subplots(figsize=(6, 6))
show_image(master_dark, "MASTER DARK (bias-subtracted)", ax)
plt.show()

# TODO (latihan): hitung piksel panas (mis. > 50 ADU pada eksposur ini)
# — berapa banyak hot pixel detektor ini?


## 5. Master Flat

Combine flats, subtract bias, then normalize so the median is 1.
Dividing by this frame corrects pixel-to-pixel sensitivity, vignetting, dust.

In [ ]:
# --- Master flat — median combine, bias subtract, normalize ---------------
flat_stack = load_fits_list(FLAT_FILES)
print("flat stack:", flat_stack.shape)

master_flat_raw = np.median(flat_stack, axis=0) - master_bias
master_flat     = master_flat_raw / np.median(master_flat_raw)   # normalize to 1.0

fig, ax = plt.subplots(figsize=(6, 6))
show_image(master_flat, "MASTER FLAT (normalized)", ax)
plt.show()

print(f"flat signal level: median raw = {np.median(master_flat_raw):.1f} ADU")

# TODO (exercise): plot a horizontal cut across the flat center.
# Where is the vignetting strongest?

## 6. Calibrate the science frame

The standard reduction:

```
Calibrated = (Science - MasterBias - Dark_rate * t_exp) / MasterFlat
```

First with **plain numpy** so the arithmetic is visible, then (after)
the same thing with **ccdproc** as a built-in alternative.

In [ ]:
# --- Version A: manual numpy ---------------------------------------------
sci_data = data.copy()                               # loaded in Section 2
sci_exptime = float(header['EXPTIME'])               # TODO: adjust keyword

sci_debiased = sci_data - master_bias
sci_darkcorr = sci_debiased - master_dark * (sci_exptime / dark_exptime)

# Avoid dividing by ~0 in bad pixels
good = master_flat > 0.5                             # TODO: pick your threshold
sci_calibrated = np.where(good, sci_darkcorr / np.where(good, master_flat, 1.0), 0.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
show_image(np.clip(sci_data, np.percentile(sci_data, 5), np.percentile(sci_data, 95)),
           "RAW", axes[0])
show_image(sci_calibrated, "CALIBRATED (manual)", axes[1])
plt.suptitle("TODO: replace titles with object/filter/epoch")
plt.show()

print(f"backgrough level: raw={np.median(sci_data[slice(0,100),slice(0,100)]):.1f} "
      f"calibrated={np.median(sci_calibrated[slice(0,100),slice(0,100)]):.1f}")

# TODO (exercise): zoom in on a 200x200 cutout containing a star before and
# after calibration (e.g. with plt.imshow on a subregion). What changed?

In [ ]:
# --- Version B: the same pipeline with ccdproc ---------------------------
# ccdproc keeps the FITS header along with the pixels, and its units/hdr keywords are handy.

import astropy.units as u

ccd_sci = ccdproc.CCDData.read(SAMPLE, unit='adu')  # TODO: unit depends on your camera

# wrap the calibration frames into CCDData first
ccd_bias = ccdproc.CCDData(master_bias, unit='adu')
ccd_dark = ccdproc.CCDData(master_dark, unit='adu')
ccd_flat = ccdproc.CCDData(master_flat, unit=u.dimensionless_unscaled)

# CCDData punya metode aritmetika (unit diurus otomatis)
ccd_sci = ccd_sci.subtract(ccd_bias)
ccd_sci = ccd_sci.subtract(ccd_dark.multiply(sci_exptime / dark_exptime))   # skala dark ke eksposur science
ccd_sci = ccd_sci.divide(ccd_flat)

# Alternatif "high-level" ccdproc:
#   ccd_sci = ccdproc.subtract_bias(ccd_sci, ccd_bias)
#   ccd_sci = ccdproc.flat_correct(ccd_sci, ccd_flat)

fig, ax = plt.subplots(figsize=(7, 7))
show_image(ccd_sci.data, "CALIBRATED (ccdproc)", ax)
plt.show()

# TODO (exercise): diff the manual and ccdproc results (sci_calibrated - ccd_sci.data).
# They should be ~identical modulo floating point. If not — can you find why?

## 7. Cosmic-ray removal

Calibration does not remove cosmic-ray events; for that use
`astroscrappy` (L.A.Cosmic algorithm, van Dokkum 2001).

In [ ]:
# --- Detect cosmic rays ---
crmask, cleaned = detect_cosmics(
    ccd_sci.data,
    sigclip=4.5,       # TODO: tune for your data quality
    invar=None,
    cleantype='medmask',
)
print(f"cosmic-ray pixels flagged: {crmask.sum()} "
      f"({100 * crmask.mean():.3f}% of the frame)")

fig, ax = plt.subplots(figsize=(7, 7))
show_image(crmask.astype(float), f"COSMIC-RAY MASK ({crmask.sum()} px)", ax)
plt.show()

# TODO (exercise): plot one flagged event zoomed in — a sharp triangle typically.
# TODO (exercise): change sigclip to 3.5 and 6, and see how the count changes.

## 8. Save the results

Write the calibrated frame back to a FITS file with a header that records
what was done (good practice — and needed if the file will be shot-processed later).

In [ ]:
# --- Save the calibrated frame ---
out_header = header.copy()
out_header['HISTORY'] = 'Calibrated in mini-workshop notebook'
out_header['HISTORY'] = 'master bias + scaled dark + flat correction'
out_header['HISTORY'] = 'cosmic rays removed with astroscrappy'

OUT_NAME = os.path.join(DATA_DIR, 'result_calibrated.fits')   # TODO: rename nicely

fits.PrimaryHDU(data=cleaned, header=out_header).writeto(OUT_NAME, overwrite=True)
print('written:', OUT_NAME)

# TODO (exercise): download it from Colab via the Files sidebar (left panel)
# or with:  from google.colab import files; files.download(OUT_NAME)

## 9. Exercises for participants (homework)

> Fill these in with tasks tailored to your class.

1. TODO — e.g. *rebuild the master flat for the other filter and compare the
   two flats pixel-by-pixel.*
2. TODO — e.g. *calibrate the science frame keeping the bad pixels masked,
   and visualise how the signal-to-noise changes.*
3. TODO — e.g. *measure the background standard deviation before and after
   flat fielding. By how much did the flat improve the noise?*

---

## References

- [`ccdproc` documentation](https://ccdproc.readthedocs.io/) — priorities and pitfalls
- [Astropy CCD data reduction notebook](https://learn.astropy.org/tutorials/)
- van Dokkum (2001), PASP 113, 1420 — L.A.Cosmic algorithm
